In [10]:
import arff
import pandas as pd
import emoji
from collections import Counter

with open("../../../00-source/OffComBR-master/OffComBR3.arff", encoding="utf-8") as f:
    dataset = arff.load(f)

cols = [attr[0] for attr in dataset["attributes"]]
df = pd.DataFrame(dataset["data"], columns=cols)

df.head()

,@@class,document
0,yes,Votaram no PEZAO Agora tomem no CZAO
1,no,cuidado com a poupanca pessoal Lembram o que a...
2,no,Sabe o que eu acho engracado os nossos governa...
3,no,Podiam retirar dos lucros dos bancos
4,no,CADE O GALVAO PRA NARRAR AGORA FALIIIIUUUUUU...


In [13]:
# Extração dos emojis
df["is_odio"] = df["@@class"].map({"yes": 1, "no": 0})
df["comentario"] = df["document"]

def extrair_emojis(texto):
    if pd.isna(texto):
        return []
    return [char for char in texto if char in emoji.EMOJI_DATA]

df["emojis"] = df["comentario"].apply(extrair_emojis)
df[["comentario", "emojis"]].head()

df_odio = df[df["is_odio"] == 1] # 1 = discurso de ódio
df_nao_odio = df[df["is_odio"] == 0] # 0 = não discurso de ódio

# Conta quantas vezes cada emoji aparece nas mensagens de ódio
emojis_odio = Counter(e for lista in df_odio["emojis"] for e in lista)

# Conta quantas vezes cada emoji aparece nas mensagens de não ódio
emojis_nao_odio = Counter(e for lista in df_nao_odio["emojis"] for e in lista)

# Junta todos os emojis encontrados nas duas categorias
todos = set(emojis_odio) | set(emojis_nao_odio)

# Montagem da tabela final
df_emojis = pd.DataFrame({
    "emoji": list(todos),
    "freq_odio": [emojis_odio.get(e, 0) for e in todos],
    "freq_nao_odio": [emojis_nao_odio.get(e, 0) for e in todos],
})

df_emojis["freq_total"] = (
    df_emojis["freq_odio"] + df_emojis["freq_nao_odio"]
)

# Normalização das frequências
total_odio = df_odio.shape[0]
total_nao_odio = df_nao_odio.shape[0]

df_emojis["freq_odio_norm"] = df_emojis["freq_odio"] / total_odio
df_emojis["freq_nao_odio_norm"] = df_emojis["freq_nao_odio"] / total_nao_odio

# Salvamento da tabela em .csv
df_emojis.to_csv(
    "twitter_emojis_fortuna2019_normalizado.csv",
    index=False
)